# Treino final de produção — LightGBM, segmentação por cluster

## Objetivo

Treinar o modelo de produção final — **LightGBM, segmentação por cluster
estatístico** — para os quatro produtos (GLP, Gasolina, Etanol, Diesel), e
exportar os artefatos necessários para o notebook de inferência
(`inferencia_previsao.ipynb`) e, posteriormente, para a aplicação Streamlit.

**Decisão de escopo:** usar a variante *cluster* para todos os produtos, por
consistência de pipeline — mesmo sabendo que, isoladamente, a segmentação
por estado teria performance ligeiramente melhor para GLP e Gasolina (ver
consolidação em `modelagem_03_LGBM_XGB.ipynb`: Estado+LGBM bateu
Cluster+LGBM em Gasolina — 0.0696 vs 0.0748 — e em GLP — 1.1441 vs 1.1556).
Essa concessão é aceita conscientemente em favor de um pipeline único e
mais simples de manter/servir.

**Nota geral já estabelecida no projeto:** nenhum modelo testado (árvores,
SARIMAX, Prophet) superou o baseline de persistência (`lag_1`) no período
de teste. O LightGBM + cluster foi escolhido por ser a opção mais simples
de operacionalizar entre as que ficaram mais próximas do baseline, **não**
por superá-lo. Essa limitação é exibida com transparência na aplicação
final (previsão do modelo lado a lado com a previsão do baseline).

In [1]:
import os
import json

import numpy as np
import pandas as pd
import lightgbm as lgb

from utils.tools_optimize import TratamentoIniciaisDF, CLUSTERS_POR_PRODUTO
from modelagem_00 import prepara_X_y, alinhar_colunas, COLUNAS_REMOVER_BASE

PRODUTOS = ["GLP", "GASOLINA", "ETANOL", "DIESEL"]

# cwd de trabalho segue a mesma convenção dos notebooks anteriores (roda a
# partir de src/); resolvemos a raiz do repo para salvar artefatos em
# models/ e resultados/ na raiz, independente de onde o notebook é aberto.
REPO_ROOT = os.getcwd()[:-4] if os.getcwd().endswith(r"\src") else os.getcwd()
PASTA_MODELOS = os.path.join(REPO_ROOT, "models")
PASTA_RESULTADOS = os.path.join(REPO_ROOT, "resultados")
os.makedirs(PASTA_MODELOS, exist_ok=True)
os.makedirs(PASTA_RESULTADOS, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)

REPO_ROOT: C:\Users\ferna\Dev\Projetos\GitHub\ANP


## 1. Carregamento e tratamento dos dados

In [2]:
path_dados = os.path.join(REPO_ROOT, "data", "dados_anp_modelado.parquet")

tratamento = TratamentoIniciaisDF(path_dados)
df_train, df_val, df_test = tratamento.apply_filters_date()

print("df_train:", df_train.shape)
print("df_val:  ", df_val.shape)
print("df_test: ", df_test.shape)

df_train: (51624, 32)
df_val:   (1944, 32)
df_test:  (5184, 32)


Para o cálculo do MAE por **estado** (não por cluster — granularidade
máxima ao usuário final, conforme decidido), precisamos de uma referência
independente com a coluna `Estado - Sigla` original, já que
`aplica_cluster_por_produto()` a substitui por `cluster`. Carregamos uma
cópia paralela do parquet bruto, aplicando exatamente a mesma leitura e
indexação que `TratamentoIniciaisDF.__init__` — sem passar pela
transformação de cluster — para poder alinhar por posição/índice com
`df_test` mais adiante.

In [3]:
df_raw = pd.read_parquet(path_dados)
df_raw.index = df_raw["dt_week"]
df_raw = df_raw.drop(columns=["dt_week"])

print(df_raw.shape)
print(df_raw["Estado - Sigla"].nunique(), "estados")

(100224, 30)
27 estados


## 2. Hiperparâmetros reaproveitados (Optuna já rodado)

**Nota de transparência importante:** os `melhores_params` abaixo foram
copiados de `resultados/metricas_por_produto_lgbm.csv`. Ao comparar os
valores de MAE desse CSV com a tabela consolidada de
`modelagem_03_LGBM_XGB.ipynb`, eles batem exatamente com a coluna
**"Estado + LGBM"** (segmentação por estado, 27 dummies) e não com a
coluna "Cluster + LGBM" — ou seja, este CSV registra o resultado da rodada
de tuning feita com segmentação por estado, não por cluster.

Mesmo assim, optou-se por reaproveitar esses hiperparâmetros como ponto de
partida para o treino por cluster (em vez de rodar Optuna novamente),
porque:
- o espaço de busca e as demais features (lags, rolling, exógenas)
  permanecem idênticos — muda apenas a granularidade da variável
  geográfica (poucas colunas de cluster vs. muitas colunas de estado);
- o custo de um novo ciclo completo de tuning bayesiano (50 trials × 4
  produtos) não se justifica frente ao ganho esperado, já que nenhuma
  configuração testada supera o baseline de qualquer forma.

Essa é uma concessão pragmática, documentada aqui para transparência
metodológica — na mesma linha das demais decisões registradas no README.

In [4]:
MELHORES_PARAMS_LGBM = {
    "GLP": {
        "max_depth": 4, "num_leaves": 223, "learning_rate": 0.012114082617941987,
        "n_estimators": 200, "subsample": 0.7446846953239656,
        "colsample_bytree": 0.6192814205163845, "min_child_samples": 15,
        "reg_alpha": 1.1200254885991283, "reg_lambda": 3.6515285309762002,
    },
    "ETANOL": {
        "max_depth": 7, "num_leaves": 41, "learning_rate": 0.034076550449029686,
        "n_estimators": 400, "subsample": 0.7214678908609284,
        "colsample_bytree": 0.5413250973712368, "min_child_samples": 33,
        "reg_alpha": 9.742700258507792, "reg_lambda": 2.674699844188844,
    },
    "DIESEL": {
        "max_depth": 8, "num_leaves": 220, "learning_rate": 0.011606368403165522,
        "n_estimators": 200, "subsample": 0.9164394060223291,
        "colsample_bytree": 0.9692926410368928, "min_child_samples": 12,
        "reg_alpha": 0.004432179913410195, "reg_lambda": 5.550858232321109,
    },
    "GASOLINA": {
        "max_depth": 6, "num_leaves": 54, "learning_rate": 0.017566375440413853,
        "n_estimators": 300, "subsample": 0.5016354351184314,
        "colsample_bytree": 0.5929476830925929, "min_child_samples": 5,
        "reg_alpha": 2.6522587482708886, "reg_lambda": 0.0011017158720671766,
    },
}

## 3. Treino do modelo de produção, por produto

Para cada produto: um **modelo de validação** (treinado só em `df_train`,
2016-01 a 2025-04 — usado na próxima seção só para gerar o MAE por
estado no bloco de teste, nunca o modelo de produção) e o **modelo de
produção** (treinado com `df_train + df_val + df_test` combinados — este
sim é exportado para a inferência).

In [5]:
modelos_producao = {}
modelos_validacao = {}
colunas_por_produto = {}

for produto in PRODUTOS:
    df_train_p = tratamento.prepara_df_produto(df_train, produto)
    df_val_p = tratamento.prepara_df_produto(df_val, produto)
    df_test_p = tratamento.prepara_df_produto(df_test, produto)

    X_train, y_train, lag1_train = prepara_X_y(df_train_p, COLUNAS_REMOVER_BASE)
    y_res_train = y_train - lag1_train
    colunas_modelo = X_train.columns.tolist()
    colunas_por_produto[produto] = colunas_modelo

    params = dict(MELHORES_PARAMS_LGBM[produto])
    params["random_state"] = 42
    params["verbosity"] = -1

    # modelo de validação — só para o MAE por estado (seção 4)
    modelo_val = lgb.LGBMRegressor(**params)
    modelo_val.fit(X_train, y_res_train)
    modelos_validacao[produto] = modelo_val

    # modelo de produção — treino + validação + teste combinados
    df_prod = pd.concat([df_train_p, df_val_p, df_test_p])
    X_prod, y_prod, lag1_prod = prepara_X_y(df_prod, COLUNAS_REMOVER_BASE)
    X_prod = alinhar_colunas(X_prod, colunas_modelo)
    y_res_prod = y_prod - lag1_prod

    modelo_prod = lgb.LGBMRegressor(**params)
    modelo_prod.fit(X_prod, y_res_prod)
    modelos_producao[produto] = modelo_prod

    print(f"{produto}: {len(colunas_modelo)} colunas | "
          f"treino={len(df_train_p)} val={len(df_val_p)} teste={len(df_test_p)}")

C:\Users\ferna\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] O sistema não pode encontrar o arquivo especificado
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\ferna\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\ferna\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ferna\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\ferna\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi

GLP: 22 colunas | treino=12906 val=486 teste=1296


GASOLINA: 22 colunas | treino=12906 val=486 teste=1296


ETANOL: 22 colunas | treino=12906 val=486 teste=1296


DIESEL: 22 colunas | treino=12906 val=486 teste=1296


## 4. Geração da tabela de MAE por produto × estado

Usa o **modelo de validação** (nunca o de produção, que já viu o bloco de
teste) para prever no bloco de teste (set/2025 a ago/2026), reconstrói o
preço (`lag_1 + resíduo previsto`) e agrupa por `produto` + `estado`
(granularidade máxima ao usuário final — não por cluster).

In [6]:
def mascara_teste(indice):
    return (
        ((indice.year == 2025) & (indice.month >= 9))
        | ((indice.year == 2026) & (indice.month <= 8))
    )


resultados_por_linha = []

for produto in PRODUTOS:
    df_test_p = tratamento.prepara_df_produto(df_test, produto)

    X_teste, y_teste_real, lag1_teste = prepara_X_y(df_test_p, COLUNAS_REMOVER_BASE)
    X_teste = alinhar_colunas(X_teste, colunas_por_produto[produto])

    pred_residuo = modelos_validacao[produto].predict(X_teste)
    preco_previsto = lag1_teste.values + pred_residuo
    preco_baseline = lag1_teste.values

    # referência independente de estado, filtrada com a mesma máscara de data
    raw_produto_teste = df_raw[
        mascara_teste(df_raw.index) & (df_raw["Produto"] == produto)
    ]
    assert len(raw_produto_teste) == len(df_test_p), (
        f"Divergência de linhas para {produto}: "
        f"{len(raw_produto_teste)} (raw) vs {len(df_test_p)} (dummy)"
    )
    assert (raw_produto_teste.index == df_test_p.index).all(), (
        f"Índices fora de ordem para {produto} — alinhamento por posição inválido"
    )
    estados = raw_produto_teste["Estado - Sigla"].values

    resultados_por_linha.append(pd.DataFrame({
        "produto": produto,
        "estado": estados,
        "preco_real": y_teste_real.values,
        "preco_previsto": preco_previsto,
        "preco_baseline": preco_baseline,
    }))

df_resultado_geral = pd.concat(resultados_por_linha, ignore_index=True)
df_resultado_geral.head()

,produto,estado,preco_real,preco_previsto,preco_baseline
0,GLP,AC,122.0,122.052137,122.0
1,GLP,AC,122.0,122.104124,122.0
2,GLP,AC,123.0,122.049089,122.0
3,GLP,AC,124.0,123.049089,123.0
4,GLP,AC,124.0,123.512612,124.0


In [7]:
mae_modelo_estado = (
    df_resultado_geral
    .assign(erro_absoluto=lambda d: (d["preco_previsto"] - d["preco_real"]).abs())
    .groupby(["produto", "estado"])["erro_absoluto"]
    .mean()
    .reset_index()
    .rename(columns={"erro_absoluto": "mae_estado"})
)

mae_baseline_estado = (
    df_resultado_geral
    .assign(erro_absoluto_baseline=lambda d: (d["preco_baseline"] - d["preco_real"]).abs())
    .groupby(["produto", "estado"])["erro_absoluto_baseline"]
    .mean()
    .reset_index()
    .rename(columns={"erro_absoluto_baseline": "mae_baseline_estado"})
)

mae_por_produto_estado = mae_modelo_estado.merge(
    mae_baseline_estado, on=["produto", "estado"]
)

mae_por_produto_estado.to_csv(
    os.path.join(PASTA_RESULTADOS, "mae_por_produto_estado.csv"), index=False
)

print(mae_por_produto_estado.shape)
mae_por_produto_estado.head()

(108, 4)


,produto,estado,mae_estado,mae_baseline_estado
0,DIESEL,AC,0.180247,0.178958
1,DIESEL,AL,0.226978,0.214583
2,DIESEL,AM,0.091659,0.051250
3,DIESEL,AP,0.090074,0.025208
4,DIESEL,BA,0.103096,0.093542


## 5. Exportação dos artefatos

In [8]:
for produto in PRODUTOS:
    sufixo = produto.lower()

    # salvo via .booster_ (API nativa do LightGBM) — o notebook de
    # inferência deve carregar com lgb.Booster(model_file=...), não com
    # LGBMRegressor(), para manter consistência com o formato salvo aqui.
    modelos_producao[produto].booster_.save_model(
        os.path.join(PASTA_MODELOS, f"modelo_producao_{sufixo}_lgbm.json")
    )

    with open(
        os.path.join(PASTA_MODELOS, f"colunas_{sufixo}_lgbm.json"), "w"
    ) as f:
        json.dump(colunas_por_produto[produto], f, indent=2)

with open(os.path.join(PASTA_RESULTADOS, "clusters_por_produto.json"), "w") as f:
    json.dump(CLUSTERS_POR_PRODUTO, f, indent=2)

print("Artefatos exportados em:", PASTA_MODELOS, "e", PASTA_RESULTADOS)

Artefatos exportados em: C:\Users\ferna\Dev\Projetos\GitHub\ANP\models e C:\Users\ferna\Dev\Projetos\GitHub\ANP\resultados


## 6. Validação final (sanity check)

In [9]:
# 1) todos os 4 modelos salvos recarregam sem erro e produzem previsão
for produto in PRODUTOS:
    sufixo = produto.lower()
    booster = lgb.Booster(
        model_file=os.path.join(PASTA_MODELOS, f"modelo_producao_{sufixo}_lgbm.json")
    )
    with open(os.path.join(PASTA_MODELOS, f"colunas_{sufixo}_lgbm.json")) as f:
        colunas = json.load(f)

    df_test_p = tratamento.prepara_df_produto(df_test, produto)
    X_teste, _, _ = prepara_X_y(df_test_p, COLUNAS_REMOVER_BASE)
    X_teste = alinhar_colunas(X_teste, colunas)
    pred = booster.predict(X_teste.head(3))
    print(f"{produto}: recarregado ok — {len(colunas)} colunas — exemplo de previsão: {pred}")

# 2) mae_por_produto_estado.csv tem 27 estados x 4 produtos = 108 linhas
assert len(mae_por_produto_estado) == 27 * 4, (
    f"Esperado 108 linhas, encontrado {len(mae_por_produto_estado)}"
)

# 3) nenhum valor nulo nas colunas de MAE
assert mae_por_produto_estado[["mae_estado", "mae_baseline_estado"]].isnull().sum().sum() == 0

print("\nSanity check OK: 108 linhas, sem nulos, 4 modelos recarregam e preveem.")

GLP: recarregado ok — 22 colunas — exemplo de previsão: [0.22179126 0.24831608 0.31887632]
GASOLINA: recarregado ok — 22 colunas — exemplo de previsão: [-0.23648576 -0.00241412 -0.01466836]
ETANOL: recarregado ok — 22 colunas — exemplo de previsão: [-0.02610642 -0.02392461 -0.0227725 ]
DIESEL: recarregado ok — 22 colunas — exemplo de previsão: [-0.37643364  0.09486725  0.04532855]

Sanity check OK: 108 linhas, sem nulos, 4 modelos recarregam e preveem.
